In [ ]:
# Parámetros (papermill overrides con -p DATA_DIR ... -p SEED ... -p OUT_DIR ...)
DATA_DIR = "./data"
SEED = 42
OUT_DIR = "reports"
MAX_SAMPLES = 50000

# 01 — Baseline Clásico: TF-IDF + Regresión Logística y SVM
**Autores:** Giuliano Crenna, Bruno Emmanuel Pace (UGR)  
**Etapa:** 4 — Modelado  

Este notebook entrena y compara los dos modelos de referencia clásicos establecidos en la metodología de la tesina:
1. **TF-IDF (1-2 gramas) + Regresión Logística** (con pesos balanceados)
2. **TF-IDF (1-2 gramas) + Support Vector Machine Lineal (LinearSVC)**

Se evalúan métricas clave (F1-macro, F1-weighted, ROC-AUC, Kappa) y se analizan los coeficientes más informativos asociados a sintomatología depresiva.

In [ ]:
import os
import sys
from pathlib import Path

# Asegurar que la raíz del repositorio esté en el sys.path
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from src.utils.seeds import set_seed
from src.models.train_baseline import load_config, load_split_data, build_baseline_pipeline, train_and_eval_model

set_seed(SEED)
OUT_DIR = Path(OUT_DIR)
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio raíz: {REPO_ROOT}")
print(f"Guardando reportes en: {OUT_DIR}")

## 1. Carga de datos de entrenamiento, validación y prueba
Se cargan los splits generados a nivel usuario en la Etapa 2 para evitar data leakage.

In [ ]:
splits_dir = Path(DATA_DIR) / "processed" / "splits"
df_train, df_val, df_test = load_split_data(
    data_dir=splits_dir,
    max_samples=MAX_SAMPLES,
    seed=SEED
)

print(f"Train: {len(df_train):,} filas | Distribución: {df_train['label'].value_counts().to_dict()}")
print(f"Val:   {len(df_val):,} filas | Distribución: {df_val['label'].value_counts().to_dict()}")
print(f"Test:  {len(df_test):,} filas | Distribución: {df_test['label'].value_counts().to_dict()}")

## 2. Entrenamiento de modelos y cálculo de métricas

In [ ]:
cfg = load_config(REPO_ROOT / "configs" / "models.yaml")

# 1. Logistic Regression
pipe_lr, res_lr = train_and_eval_model(
    model_type="logreg",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    config=cfg,
    seed=SEED
)

# 2. Linear SVM
pipe_svm, res_svm = train_and_eval_model(
    model_type="svm",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    config=cfg,
    seed=SEED
)

## 3. Comparación de Rendimiento en Test

In [ ]:
rows = []
for m_name, res in [("Logistic Regression", res_lr), ("Linear SVM", res_svm)]:
    test_m = res["test"]
    rows.append({
        "Modelo": m_name,
        "Accuracy": test_m["accuracy"],
        "F1 Macro": test_m["f1_macro"],
        "F1 Weighted": test_m["f1_weighted"],
        "Cohen Kappa": test_m["kappa"],
        "AUC (si aplica)": test_m.get("auc_ovr_macro", np.nan)
    })

df_res = pd.DataFrame(rows)
df_res.to_csv(TAB_DIR / "baseline_summary.csv", index=False)
print(df_res.to_markdown(index=False))

## 4. Matrices de Confusión (Evaluación de Falsos Negativos y Riesgo Clínico)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
y_true = df_test["label"].astype(int).tolist()
labels_unique = sorted(list(set(y_true)))

for idx, (m_name, pipe) in enumerate([("Logistic Regression", pipe_lr), ("Linear SVM", pipe_svm)]):
    y_pred = pipe.predict(df_test["text_clean"].tolist())
    cm = confusion_matrix(y_true, y_pred, labels=labels_unique)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=labels_unique, yticklabels=labels_unique)
    axes[idx].set_title(f"Matriz de Confusión — {m_name}")
    axes[idx].set_xlabel("Predicción")
    axes[idx].set_ylabel("Etiqueta Real")

plt.tight_layout()
fig_path = FIG_DIR / "baseline_confusion_matrix.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Figura guardada en {fig_path}")

## 5. Interpretabilidad del Baseline: Términos y n-gramas más predictivos
Se extraen los pesos del clasificador de Regresión Logística para identificar qué términos empujan con mayor fuerza hacia la clase depresiva vs control.

In [ ]:
vec = pipe_lr.named_steps["tfidf"]
clf = pipe_lr.named_steps["clf"]
feature_names = np.array(vec.get_feature_names_out())
coefs = clf.coef_[0]

top_dep = np.argsort(coefs)[-15:]
top_ctrl = np.argsort(coefs)[:15]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#d9534f" if c > 0 else "#5bc0de" for c in np.concatenate([coefs[top_ctrl], coefs[top_dep]])]
words = np.concatenate([feature_names[top_ctrl], feature_names[top_dep]])
vals = np.concatenate([coefs[top_ctrl], coefs[top_dep]])

y_pos = np.arange(len(words))
ax.barh(y_pos, vals, color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels(words)
ax.set_xlabel("Coeficiente en Regresión Logística")
ax.set_title("Top 15 Términos Predictivos: Control (azul) vs Depresión (rojo)")
plt.tight_layout()

top_fig_path = FIG_DIR / "baseline_top_features.png"
plt.savefig(top_fig_path, dpi=300)
plt.show()
print(f"Figura guardada en {top_fig_path}")